# Pre- vs. post-SHINE cohort comparison: data quality

Is the data from the pre-SHINE and post-SHINE cohorts *equally good*?<br>
Compares within-subject test-retest reliability and how far each cohort's pairwise SpAM distances sit from a random-placement null, for **retained subjects only** (production cohort, `status == "full data"`, `task_version >= 4.0`;
same selection as `behavior.ipynb`).<br>
Companion notebooks:
- `1_demographics_and_behavior.ipynb` - do they behave the same?
- `3_geometry.ipynb` - do they give the same similarity structure?

Data-prep and figure helpers live in `data_quality.py`.

All pairwise comparisons are two-sided Mann-Whitney U tests on per-subject aggregates (one value per
subject, avoiding pseudo-replication across trials), with rank-biserial effect size
*r* = 1 - 2U/(n_A x n_B); |r| >= 0.5 is conventionally large. Colours are the validated
colorblind-safe blue/orange pair used in `analysis/pilot/figures.py`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/behavioral/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, wasserstein_distance, wilcoxon

from analysis.utils.stats import mwu_effect_size, print_comparison_line
from analysis.utils.equivalence import (
    equivalence_sensitivity, equivalence_test_rank_biserial, print_equivalence_line, resolvable_bound,
)
from analysis.utils.plotting import fig_equivalence_forest
from analysis.utils.repeat_reliability import repeat_reliability_by_index
from analysis.pilot.simulate_null_distances import simulate as simulate_null_distances
from analysis.behavioral.demographics_and_behavior import COHORTS, COHORT_LABEL, load_retained_cohorts
from analysis.behavioral.data_quality import (
    subject_reliability, subject_repeat_slope, pairwise_distance_values, subject_wasserstein_distance, subject_ks_distance,
    fig_reliability, fig_repeat_reliability, fig_distance_vs_null,
)

## 0. Load data

In [2]:
cache_dir = REPO_ROOT / "analysis" / "results" / "parsed_data"  # override if running from a worktree without this cache copied in
cohorts = load_retained_cohorts(cache_dir)
df_participants, df_trials = cohorts["df_participants"], cohorts["df_trials"]
participants, participant_ids, trials_main = cohorts["participants"], cohorts["participant_ids"], cohorts["trials_main"]

for c in COHORTS:
    print(f"{COHORT_LABEL[c]}: {len(participants[c])} retained subjects, {len(trials_main[c])} main trials")

Pre-SHINE: 79 retained subjects, 1738 main trials
Post-SHINE: 76 retained subjects, 1672 main trials


## 1. Within-subject reliability

Distribution of within-subject reliability (per-subject mean Spearman r across trial repeats)
for each cohort, and a Mann-Whitney U test of whether the two cohorts differ.

In [3]:
reliability = {c: subject_reliability(df_trials, participant_ids[c]) for c in COHORTS}

reliability_summary = pd.DataFrame({
    COHORT_LABEL[c]: {
        "median": round(float(reliability[c].median()), 3),
        "mean": round(float(reliability[c].mean()), 3),
        "SEM": round(float(reliability[c].std(ddof=1) / np.sqrt(len(reliability[c]))), 4),
    }
    for c in COHORTS
}).T
reliability_summary

,median,mean,SEM
Pre-SHINE,0.276,0.319,0.0168
Post-SHINE,0.291,0.312,0.0139


In [4]:
fig_reliability(reliability).show()

In [5]:
result_reliability = mwu_effect_size(reliability["pre"], reliability["post"], label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"])
print_comparison_line(result_reliability)
pd.DataFrame([result_reliability])

  Pre-SHINE vs Post-SHINE: U=2898.0  p=0.7110 (not significant)  r=0.035


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
0,Pre-SHINE vs Post-SHINE,79,76,0.2762,0.2912,0.3192,0.312,0.1495,0.121,2898.0,0.711035,0.0346


## 2. Reliability over repeat index

Each retained subject saw 4 verbatim trial repeats, at fixed positions in the session (about 16%, 28%,
64% and 84% of the way through) and interleaved with other trials. `repeat_index` orders them within a
subject, so a trend over repeats is a trend in test-retest reliability over the session. The figure shows
mean +/- SE reliability per repeat with thin per-subject lines; below, each subject's least-squares slope
of reliability on repeat index (r units per repeat, subjects with >= 3 valid repeats) is tested against 0
within each cohort (Wilcoxon signed-rank) and compared between cohorts (Mann-Whitney U).

In [6]:
repeat = {
        c: repeat_reliability_by_index(df_trials[df_trials["participant_id"].isin(participant_ids[c])])
        for c in COHORTS
}
fig_repeat_reliability(repeat).show()

pd.DataFrame({
    COHORT_LABEL[c]: repeat[c].groupby("repeat_index")["spearman_r"].mean().round(3) for c in COHORTS
})

,Pre-SHINE,Post-SHINE
repeat_index,,
1,0.332,0.297
2,0.306,0.324
3,0.319,0.296
4,0.321,0.331


In [7]:
repeat_slope = {c: subject_repeat_slope(repeat[c]) for c in COHORTS}
for c in COHORTS:
    _, p = wilcoxon(repeat_slope[c])
    print(f"  {COHORT_LABEL[c]} slope vs 0 (Wilcoxon): n={len(repeat_slope[c])}  median slope={repeat_slope[c].median():.4f}  p={p:.4f}")

result_slope = mwu_effect_size(
    repeat_slope["pre"], repeat_slope["post"], label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"],
)
print_comparison_line(result_slope)
pd.DataFrame([result_slope], index=["Reliability slope per repeat"])

  Pre-SHINE slope vs 0 (Wilcoxon): n=79  median slope=0.0050  p=0.9493
  Post-SHINE slope vs 0 (Wilcoxon): n=76  median slope=0.0153  p=0.3102
  Pre-SHINE vs Post-SHINE: U=2778.0  p=0.4237 (not significant)  r=0.075


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
Reliability slope per repeat,Pre-SHINE vs Post-SHINE,79,76,0.005,0.0153,-0.002,0.0076,0.0802,0.0749,2778.0,0.423716,0.0746


## 3. Pairwise distance distributions vs. the random-assignment null

Per-cohort distribution of pairwise SpAM distances (main trials, all retained subjects) against
the null distribution of pairwise distances under uniformly-random placement (same simulation as
`analysis/pilot/simulate_null_distances.py`, `num_dots=20` matching `images_per_trial`). Distance
from null is quantified two ways: **Wasserstein (earth mover's) distance is the primary measure**
-- it integrates over the full gap between the trial's distance distribution and the null, so it's
sensitive to *how far apart* the two distributions are, not just whether their shapes differ. **KS
distance (D statistic) is kept as a secondary/shape-robustness check** (it only captures the
maximum CDF gap). Both are compared between cohorts with a Mann-Whitney U test on per-subject means.

We also check, descriptively, whether each cohort's *pooled* distances actually differ from the
null at all (not just from each other) via a single Wasserstein distance + KS test per cohort
against the null sample. This pools all of a cohort's trial-level pairwise-distance observations
into one sample, so the resulting KS p-value is not a valid independent-samples test (observations
within a trial/subject aren't independent) -- it's reported as a directional/magnitude check, not
a formal inference.

In [8]:
null_distribution = simulate_null_distances(num_dots=20, num_trials=1000, seed=42)
dists = {c: pairwise_distance_values(trials_main[c]) for c in COHORTS}
wasserstein_dist = {c: subject_wasserstein_distance(trials_main[c], null_distribution) for c in COHORTS}
ks_distance = {c: subject_ks_distance(trials_main[c], null_distribution) for c in COHORTS}

fig_distance_vs_null(dists, null_distribution, wasserstein_dist, ks_distance).show()

In [9]:
for c in COHORTS:
    w = wasserstein_distance(dists[c], null_distribution)
    D, p = ks_2samp(dists[c], null_distribution)
    print(f"  {COHORT_LABEL[c]} vs null (pooled, descriptive): Wasserstein={w:.4f}  KS D={D:.4f}  KS p={p:.2e}")

  Pre-SHINE vs null (pooled, descriptive): Wasserstein=0.0274  KS D=0.0553  KS p=4.75e-321
  Post-SHINE vs null (pooled, descriptive): Wasserstein=0.0277  KS D=0.0589  KS p=0.00e+00


In [10]:
result_wasserstein = mwu_effect_size(wasserstein_dist["pre"], wasserstein_dist["post"], label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"])
result_ks = mwu_effect_size(ks_distance["pre"], ks_distance["post"], label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"])
print("Primary (Wasserstein):")
print_comparison_line(result_wasserstein)
print("Secondary (KS):")
print_comparison_line(result_ks)
pd.DataFrame([result_wasserstein, result_ks], index=["Wasserstein (primary)", "KS (secondary)"])

Primary (Wasserstein):
  Pre-SHINE vs Post-SHINE: U=2784.0  p=0.4363 (not significant)  r=0.073
Secondary (KS):
  Pre-SHINE vs Post-SHINE: U=2805.0  p=0.4818 (not significant)  r=0.066


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
Wasserstein (primary),Pre-SHINE vs Post-SHINE,79,76,0.0400,0.0412,0.0454,0.0476,0.0218,0.0231,2784.0,0.436265,0.0726
KS (secondary),Pre-SHINE vs Post-SHINE,79,76,0.1307,0.1327,0.1421,0.1463,0.0548,0.0557,2805.0,0.481839,0.0656


## 4. Equivalence tests

Every "no difference" above is a non-significant Mann-Whitney test, which is absence of evidence rather
than evidence of equivalence. As in `1_demographics_and_behavior.ipynb` (section 5), this section tests equivalence
directly (TOST in its confidence-interval form, `analysis/utils/equivalence.py`): rank-biserial *r* with a
subject-level bootstrap 90% CI, equivalent when the CI lies inside +/- a bound.

The bound is not chosen by hand. There is no pre-defined smallest effect size of interest for this
project, so every bound here follows one rule: the smallest bound with 80% power when the groups truly do
not differ (a simulation at the observed sample sizes, `analysis/utils/equivalence.py`), rounded up. A
bound tighter than that would fail to show equivalence even for identical groups too often to be
informative.

In [11]:
n_pre, n_post = len(reliability["pre"]), len(reliability["post"])
rank_sensitivity = equivalence_sensitivity(n_pre, n_post)
rank_bound = resolvable_bound(rank_sensitivity)
print(f"Rank-biserial: 80%-power sensitivity at n=({n_pre}, {n_post}) = {rank_sensitivity:.3f} -> bound +/-{rank_bound:.2f}")

Rank-biserial: 80%-power sensitivity at n=(79, 76) = 0.269 -> bound +/-0.30


In [12]:
quality_outcomes = {
    "Within-subject reliability": (reliability["pre"], reliability["post"]),
    "Reliability slope per repeat": (repeat_slope["pre"], repeat_slope["post"]),
    "Wasserstein distance from null": (wasserstein_dist["pre"], wasserstein_dist["post"]),
    "KS distance from null": (ks_distance["pre"], ks_distance["post"]),
}
equivalence_quality = {
    name: equivalence_test_rank_biserial(
        pre, post, bound=rank_bound, label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"],
    )
    for name, (pre, post) in quality_outcomes.items()
}
for name, result in equivalence_quality.items():
    print(name)
    print_equivalence_line(result)
fig_equivalence_forest(equivalence_quality, x_title="Rank-biserial r (Pre vs Post), 90% CI", title="Equivalence: data-quality outcomes").show()

Within-subject reliability
  Pre-SHINE vs Post-SHINE: effect=0.035  90% CI [-0.117, 0.190]  bound=+/-0.3 (equivalent)
Reliability slope per repeat
  Pre-SHINE vs Post-SHINE: effect=0.075  90% CI [-0.076, 0.230]  bound=+/-0.3 (equivalent)
Wasserstein distance from null
  Pre-SHINE vs Post-SHINE: effect=0.073  90% CI [-0.076, 0.225]  bound=+/-0.3 (equivalent)
KS distance from null
  Pre-SHINE vs Post-SHINE: effect=0.066  90% CI [-0.083, 0.217]  bound=+/-0.3 (equivalent)
